---
## Initialisation

---


**Les deux cellules suivantes peuvent n'être exécutées qu'une fois en début de session.**

La première cellule installe et importe les librairies utiles et définie les fonctions qui géreront :

* 1. Connexion du programme python à votre drive google
* 2. Importation des données utilisées par l'IA
* 3. Initialisation des dossiers utiles

Dans la seconde cellule, le chemin vers votre google drive est passé en variable et les fonctions précédements créées sont appelées.


In [ ]:
# @title
print("Installation de librosa...")
!pip install librosa &> /dev/null

print("Importation des librairies...")

import numpy as np # Classique, utile pour manipuler des tableaux, des matrices
from matplotlib import pyplot as plt # Permet de générer des figures
import random # Permet de générer des valeurs aléatoires (entier entre 0 et n, flottant entre 0 et 1, liste de nombres aléatoires...)
import itertools # Permet d'agir de manière itératives sur des données structurées

import tensorflow as tf # Librairie de référence dans la programmation d'IA avec python

import keras # Librairie de référence dans la programmation de réseau de neurones avec python
from keras.models import Sequential # Permet d'arranger les différentes couches du réseau de neurones
from keras.models import Model  # Permet de construire un modèle de réseau de neurones, de l'entraîner et de l'exploiter
from keras.layers import Dense # Pour ajout d'une couche dense
from keras.layers import Flatten # Pour ajout d'une couche de flatten
from keras.layers import Dropout # Pour ajout d'une couche de dropout (utilité : permet desfois d'éviter l'overfitting).
from keras.layers import Activation # Importe les différences fonction d'activation ('relu','sigmoid','sofmax',...)
from keras.layers import Conv2D # Pour ajout d'une couche de convolution 2D
from keras.layers import MaxPooling2D #Pour ajout d'une couche de MaxPooling 2D
from keras.utils import to_categorical # A vous de trouver !

from sklearn.metrics import confusion_matrix # Permet d'obtenir la matrice de confusion du modèle en croisant les labels avec les prédictions du réseau

import os # Utile pour naviguer à travers les différents dossier
from google.colab import drive # Utile pour connecter le notebook à un drive google
from pathlib import Path # Facilite la manipulation des chemins vers des fichiers

from IPython.display import Audio # Permet la lecture et l'enregistrement de fichier audio directement depuis le notebook

import librosa # Bibliothèque utilisée pour l'analyse et le traitement des données audio.

import glob # librairie spécialisée dans la recherche de chemin

from tqdm.auto import tqdm  # Fournit une barre de progression visuelle pour les boucles dans les notebooks Jupyter.

print("Installation et importation terminées !\n \n")

def connexion_drive(working_dir):
  print("Connexion au drive...")
  # point de montage de google drive dans google colab
  drive.mount('drive', force_remount=True)
  # création du dossier de travail si celui-ci n'existe pas
  if not os.path.exists(working_dir):
    os.mkdir(working_dir)
  print("Connecté au drive !")

def initialisation_dossier_melspec(data_dir):
  meldir = str(data_dir)+"/melspectrograms/"
  if not os.path.exists(meldir):
    os.mkdir(meldir)
  else:
    for elt in os.listdir(meldir):
      os.remove(meldir + elt)

def importation_des_donnees(data_dir):
  print("Importation des données...")
  # Si le dossier de données n'existe pas, on le créé
  freespoken_Dir = data_dir + "/free-spoken-digit-dataset"
  if not os.path.exists(data_dir):
    os.mkdir(data_dir)
  #téléchargement du jeu de données free-spoken-digit-dataset sur votre drive
  !git clone https://gitlab.ensam.eu/QCons/pjt1a_projetaudio $data_dir/free-spoken-digit-dataset
  !rm -rf $data_dir/free-spoken-digit-dataset/.git


Installation de librosa...
Importation des librairies...
Installation et importation terminées !
 



In [ ]:
# @title
################################################################################
# renseigner le chemin vers le dossier de travail
# dans lequel seront déposés les datas et les résultats : jeux de poids, ...
# Il reste juste à choisir le nom que vous voulez pour le working_dir (chemin)
# (i.e. renommer 2023-2024-S6-Projet-Audio)
#
# Utilisation du Drive sous Google Colab cliquer sur
# l'icone orange en haut à gauche permet d'accéder au Drive
# dans lequel vous pouvez créer des dossiers, stocker des données : dataset, poids d'un réseau entrainé, ...
################################################################################
working_dir = 'drive/MyDrive/<your_folder>'
data_dir=working_dir+"/Datasets"

# Utilisation des fonctions d'initialisation
connexion_drive(working_dir)

Connexion au drive...
Mounted at drive
Connecté au drive !


In [ ]:
ClonageDataset=False
if ClonageDataset :
  importation_des_donnees(data_dir)
# Une fois importation_des_donnees() faite dans votre Drive, vous pouvez passer cette variable ClonageDataset à False.
# Cela evitera à chaque compilation de cloner à nouveau le dataset dans le Drive
initialisation_dossier_melspec(data_dir)

Importation des données...
fatal: Too many arguments.

usage: git clone [<options>] [--] <repo> [<dir>]

    -v, --verbose         be more verbose
    -q, --quiet           be more quiet
    --progress            force progress reporting
    --reject-shallow      don't clone shallow repository
    -n, --no-checkout     don't create a checkout
    --bare                create a bare repository
    --mirror              create a mirror repository (implies bare)
    -l, --local           to clone from a local repository
    --no-hardlinks        don't use local hardlinks, always copy
    -s, --shared          setup as shared repository
    --recurse-submodules[=<pathspec>]
                          initialize submodules in the clone
    --recursive ...       alias of --recurse-submodules
    -j, --jobs <n>        number of submodules cloned in parallel
    --template <template-directory>
                          directory from which templates will be used
    --reference <repo>    refere

---
## Définition des fonctions de base

---

Dans la cellule suivante, différentes fonctions sont définies.
Le tableau ci-dessous en donne un récapitulatif


| Nom de la fonction   | Paramètre(s) | Rôle  |
|---|---|---|
| summarize_diagnostics  |  history <br> directory <br> filename | tracé de l'évolution de la loss et de l'accuracy au cours des époques d'entraînement <br> (à la fois sur les données de train et de test)  |
| confusion_matrix       |  Ylabels <br> Ypredictions            | tracé personnalisé d'une matrice de confusion <br>= outil d'analyse des prédictions pour un problème de classification  |
| occurencesclasses      |  label <br> numclasses                |  calcul du nombre d'occurences de chaque classe dans un dataset|

<br>

----

<br>

Pour rappel, la matrice de confusion est un outil agréable de représentation et d'analyse des résultats pour un problème de classification

La diagonale d'un matrice de confusion correspond donc aux bonnes prédictions, et elle permet en particulier de voir si des confusions se produisent plus spécifiquement entre certaines classes.
En recherchant sur internet (par exemple "confusion matrix Keras", "confusion matrix plot"), vous trouverez de nombreux sites sur le sujet, qui vous présentent cet outils et vous montre comment tracer une matrice de confusion et personnaliser ce tracé à vos souhaits. Par exemples :

https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics (section 3.3.2.6)

https://medium.com/@dtuk81/confusion-matrix-visualization-fc31e3f30fea

https://www.kaggle.com/grfiv4/plot-a-confusion-matrix

In [ ]:
# @title
def summarize_diagnostics(history,
                          directory,
                          filename):
    """
    pour visualiser l'entrainement du réseau,
    nous avons défini une procédure qui crée un graphique représentant l'évolution, au cours des époques :
      - Des valeurs de la fonction de perte (loss)
      - De la précision de classification (accuracy) pour les données d'entrainement et de test.

    Arguments
    ---------
    history:          historique de l'entraînement

    directory:        répertoire de sauvegarde des images

    filename:         Nom attribuée à l'image dans le dossier "directory"

    Usage
    -----
    summarize_diagnostics(history,
                          directory,
                          filename)



    """
    # plot loss
    plt.subplot(211)
    plt.title('Cross Entropy Loss')
    plt.plot(history.history['loss'], color='blue', label='train')
    plt.plot(history.history['val_loss'], color='orange', label='test')
    plt.legend()
    # plot accuracy
    plt.subplot(212)
    plt.title('Classification Accuracy')
    plt.plot(history.history['accuracy'], color='blue', label='train')
    plt.plot(history.history['val_accuracy'], color='orange', label='test')
    plt.legend()
    # save plot to file
    plt.savefig("{}/{}_plot.png".format(directory,filename))
    plt.show()
    plt.close()


def plot_confusion_matrix(cm,
                          target_names,
                          title='Confusion matrix',
                          cmap=None,
                          normalize=False):
    """
    given a sklearn confusion matrix (cm), make a nice plot

    Arguments
    ---------
    cm:           confusion matrix from sklearn.metrics.confusion_matrix

    target_names: given classification classes such as [0, 1, 2]
                  the class names, for example: ['high', 'medium', 'low']

    title:        the text to display at the top of the matrix

    cmap:         the gradient of the values displayed from matplotlib.pyplot.cm
                  see http://matplotlib.org/examples/color/colormaps_reference.html
                  plt.get_cmap('jet') or plt.cm.Blues

    normalize:    If False, plot the raw numbers
                  If True, plot the proportions

    Usage
    -----
    plot_confusion_matrix(cm           = cm,                  # confusion matrix created by
                                                              # sklearn.metrics.confusion_matrix
                          normalize    = True,                # show proportions
                          target_names = y_labels_vals,       # list of names of the classes
                          title        = best_estimator_name) # title of graph

    Citiation
    ---------
    http://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html

    """

    accuracy = np.trace(cm) / float(np.sum(cm))
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names, rotation=45)
        plt.yticks(tick_marks, target_names)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]


    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if normalize:
            plt.text(j, i, "{:0.4f}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
        else:
            plt.text(j, i, "{:,}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")


    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label\naccuracy={:0.4f}; misclass={:0.4f}'.format(accuracy, misclass))
    plt.show()

def occurencesclasses(label,numclasses):
  classes_list=np.arange(0,numclasses) # classes={0,1,...,N-1} pour un problème à N classes
  occurences=np.zeros(len(classes_list),int)
  for i in range(len(label)):
      classe=int(label[i])
      occurences[classe]=occurences[classe]+1
  return(occurences)

---
## Enregistrement et sauvegarde de fichier audio

---

Dans la cellule suivante, sont définies les fonctions du tableau ci-dessous :

| Nom de la fonction   | Paramètre(s) | Rôle  |
|---|---|---|
| button_record | | inutile pour l'utilisateur, est appelée par d'autre fonction |
| record_audio | nom, dossier, duree | enregistre un fichier audio de "duree" secondes <br> sous le nom "nom.wav" dans le dossier "dossier" (par défaut data_dir)

In [ ]:
#@title

# source ayant servi de base : https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be
# Permet d'enregistrer un fichier audio sur colab en utilisant le microphone par défaut de l'ordinateur.
from IPython.display import Javascript, Audio
from google.colab import output
from base64 import b64decode
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def button_record(filename='audio.wav',time=5):
  js = Javascript('''
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})

var record  = time => new Promise(async resolve => {

    const div = document.createElement('div');
    const capture = document.createElement('button');
    capture.textContent = 'Capture';
    div.appendChild(capture);
    document.body.appendChild(div);

    stream = await navigator.mediaDevices.getUserMedia({audio: true});

    recorder = new MediaRecorder(stream)
    chunks = []
    recorder.ondataavailable = e => chunks.push(e.data)

    await new Promise((resolve) => capture.onclick = resolve);
    recorder.start()
    capture.textContent = "Recording"
    capture.style.color = "green"
    await sleep(time)
    recorder.onstop = async ()=>{
    blob = new Blob(chunks);
    text = await b2text(blob);
    resolve(text);
  }
    stream.getAudioTracks()[0].stop();
    recorder.stop();
    div.remove();
})
''')
  display(js)
  data = eval_js('record({})'.format(time*1000))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

# Modifier le chemin du dossier record_rep ci-dessous comme vous le souhaitez.
# C'est l'endroit du drive où seront sauvegardés les enregistrements sous un nom de fichier .wav spécifié dans reord_path.

def record_audio(nom,dossier,duree=3):
  if not os.path.exists(dossier):
      os.mkdir(dossier)
  record_path= "{d}/{n}.wav".format(d=dossier,n=nom)
  # lancement de l'enregistrement
  button_record(filename=record_path,time=duree)
  # écouter l'enregistrement
  Audio(record_path)

def load_audio(nom,dossier):
  y, sr = librosa.load("{d}/{n}".format(d=dossier, n=nom), sr= 8000)
  #y est le signal sous forme d'array numpy
  # sr est le taux d'échantillonage (vaut 8khz  d'après les indications github)
  print(type(y),y, sr)
  Audio(y,rate=sr)
  return y, sr

Démonstration de la fonction d'enregistrement de l'audio suivie d'une lecture du même fichier.

Vous pouvez écouter un signal audio en utilisant load_audio() et Audio()

In [ ]:
data_record=data_dir+"/MyRecords"
# Test de la fonction enregistrement sur un PC équipé d'un microphone
# Commenter les trois lignes ci-dessous si vous ne voulez pas faire cela à chaque compilation

record_audio("demo_enregistrement",data_record,duree=2)
y, sr = load_audio("demo_enregistrement.wav",data_record)
Audio(y,rate=sr)

---
## Analyse des fichiers audio

---

Les fonctions définies dans cette cellule font appel à la librairie librosa.

Un [Tutoriel sur Librosa](https://librosa.org/doc/main/tutorial.html)

Nous pouvons calculer certaines caractéristiques des signaux audios, par exemple le [melspectrogramme](https://medium.com/analytics-vidhya/understanding-the-mel-spectrogram-fca2afa2ce53) qui est donné sous la forme de tableaux numpy. Il existe de nombreux autres [features librosa](https://librosa.org/doc/latest/feature.html) (onglet "features extraction" à gauche dans le tutoriel) pour représenter un signal audio.


Dans la cellule suivante, sont définies les fonctions du tableau ci-dessous :

| Nom de la fonction   | Paramètre(s) | Rôle  |
|---|---|---|
| plot_spec | S_dB | Affiche le melspec|
| compute_spectrograme | path | Calcule le melspec|
| get_duration | spec  | Récupère la longueur du fichier audio |
| get_longest_duration | list_path <br> spec | Trouve la plus longue durée de fichier audio d'un dossier |
| save_melspec | path_melspec | Sauvegarde le melspec |


In [ ]:
def plot_spec(S_dB):
  plt.figure(figsize=(10, 4))
  librosa.display.specshow(S_dB, x_axis='time',
                         y_axis='mel', sr=sr)
  plt.colorbar(format='%+2.0f dB')
  plt.title('Mel-frequency spectrogram')
  plt.tight_layout()
  plt.show()

def compute_spectrogram(path):
    "Calcule le melspectrogramme du fichier audio situé à path dans le repertoire de fichiers"
    file_name = os.path.basename(path)
    s , _ = librosa.load(path, sr= sr) # sr est le taux d'échantillonnage du signal.

    melspec = librosa.feature.melspectrogram(y=s, sr=sr, n_mels=n_mels,n_fft=1024)
    melspec_db = librosa.power_to_db(melspec, ref=np.max)
    # Nous rajoutons une dimension 1 à la fin pour forcer la représentation d'une image en tant que tenseur
    # (hauteur, largeur, nombre de canaux ici égal à 1)
    # pour compatibilité avec la première couche Conv2D du réseau de neurones
    melspec_db_image = melspec_db.reshape(melspec_db.shape[0],melspec_db.shape[1],1)
    return (path,melspec_db_image)

def get_duration(spec):
  "Renvoie la dimension correspondant à l'axe temporel d'un melspectrogramme"
  return spec.shape[1]

def get_longest_duration(list_path_spec):
  "Renvoie la dimension temporelle du melspectrogramme le plus long"
  list_time = [get_duration(spec) for path, spec in list_path_spec ]
  return max(list_time)

def save_melspec(path_melspec):
    """
    Permet d'enregistrer les spectrogrammes dans le dossier melspectrogram
    """
    path, melspec = path_melspec
    file_name = os.path.basename(path)
    melspec_path = os.path.join(data_dir,"melspectrograms", os.path.splitext(file_name)[0])
    np.save(melspec_path, melspec)
    #print("{} saved".format(melspec_path))

def save_fft(path_melspec):
  print(path_melspec)
  pass

In [ ]:
sr = 8000

# Génération d'un sinus à 512 HZ
y1 = 100*np.sin(2 * np.pi * 512 * np.linspace(0,2.5,int(8000*2.5)))
y2 = np.sin(2 * np.pi * 1024 * np.linspace(0,2.5,int(8000*2.5)))

y = np.concatenate([y1,y2])

sp = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128) # calcul du melspectrogramme
s_dB = librosa.power_to_db(sp, ref=np.max) #normalisation pour meilleure visualisation

print(type(s_dB))

plot_spec(s_dB)

<class 'numpy.ndarray'>



---

# Création/Sauvegarde des mel-spectrogrammes pour free-spoken-digit-dataset

---

Pour éviter d'avoir à recalculer constamment les mel-spectrogrammes, il vaut mieux les calculer une fois et les sauvegarder dans le dossier melspectrogram. Nous commençons par charger les chemins de fichiers.

In [ ]:
list_of_files = glob.glob(data_dir+"/recordings/*") #permet d'obtenir le chemin relatif de tous les fichiers du dossier recordings


i=3 # Application sur un des fichiers
print(list_of_files)
# affichage du nom d'un des fichers
# la nomenclature des fichiers du dataset est chiffreprononcé_nomduspeaker_occurence.wav où occurence={0,...,49}
print(list_of_files[i])

n_files = len(list_of_files) # nombres de fichiers audios (3000 pour vérification)
print(n_files)

x, sr = librosa.load(list_of_files[i], sr= 8000) #y est le signal sous forme d'array numpy, sr est le taux d'échantillonage (vaut 8khz  d'après les indications github)
print(type(x),np.shape(x),sr)

#lecture du fichier
Audio(x,rate=sr)

# création et affichage du mel-spectrogramme
sp = librosa.feature.melspectrogram(y=x, sr=sr, n_mels=128) # calcul du melspectrogramme
s_dB = librosa.power_to_db(sp, ref=np.max)
plot_spec(s_dB)

n_mels=128
sr = 8000

###Prétraitement et enregistrement des fichiers audios en mel-spectrogramme


Définition des fonctions utilisées pour le prétraitement

In [ ]:
n=n_files

# la création des mel-spectrogrammes prend un peu de temps
# Pour éviter d'avoir à recalculer constamment les mel-spectrogrammes,
# nous les sauvegardons dans le dossier melspectrogram du DRIVE.
# une fois cela fait (cette cellule compilée et donc les mel-spectrogammes crées et sauvegardés dans le drive), vous pouvez passer donc la variable boolenne
# la variable CreationSpectrogrammes à False pour éviter à nouveau cette procédure
CreationSpectrogrammes=False

if CreationSpectrogrammes :

  print("Conversion des fichiers audio en melspectrogrammes")
  list_spec = []
  for path, spec in tqdm(map(compute_spectrogram, list_of_files[:n]), total=n):
      list_spec.append((path, spec))

  max_time = get_longest_duration(list_spec)
  print("La plus longue dimension temporelle est {}".format(max_time))
  def spec_padding(path_data):
    path, spec = path_data
    min_spec = np.min(spec)
    spec_padded = np.pad(spec, ((0,0),(0,max_time-spec.shape[1]),(0,0)),"constant",constant_values=((min_spec,min_spec),(min_spec,min_spec),(min_spec,min_spec)))
    return (path, spec_padded)


  print("Traitement des melspectrogrammes pour qu'ils aient la même durée")
  list_padded_spec = []
  for path, spec_padded in tqdm(map(spec_padding, list_spec[:n]), total=n):
      list_padded_spec.append((path,spec_padded))

  for spec in tqdm(map(save_melspec, list_padded_spec[:n]), total=n):
    pass


---

# Partie Travail Commun

---

## Labélisation des données

Organiser votre donnée pour pouvoir entraîner un réseau de neurones. Vous devriez avoir en sortie :
- une liste ou array **X** comprenant les images
- une liste ou array **Y_speaker** contenant le nom des speaker, une liste ou array **Y_chiffre** contenant le chiffre prononcé, et une liste ou array **Y_occ** contenant le N° d'occurence (ce "label" sera utile pour splitter facilement vos données en train et test de manière homogène)
 OU une liste de liste ou array **Y=[Y_speaker,Y_chiffre,Y_occ]** comprenant les 3 labels   

Le petit bout de code
```
os.listdir(datamelspec)
```

fournit permet de lister/charge tous les fichiers présents dans le dossier melspectrogram (pour éviter d'avoir à recalculer constamment les mel-spectrogrammes, nous les avons sauvegardés dans le dossier melspectrogram).


In [ ]:
import os
import numpy as np

# Liste des speakers
Lspeaker = ['jackson', 'nicolas', 'theo', 'yweweler', 'george', 'lucas']

# Dossier contenant les melspectrograms
datamelspec = data_dir + '/melspectrograms'

# Initialisation des listes
X = []
Y_speaker = []
Y_chiffre = []
Y_occ = []

for f in os.listdir(datamelspec):
    if f.endswith('.npy'):
        data = f.split('.')[0].split('_')

        chiffre = int(data[0])             # chiffre prononcé
        speaker = data[1]                  # nom du speaker
        occ = int(data[2])                 # n° d’occurrence

        # Chargement du melspectrogramme
        mel = np.load(os.path.join(datamelspec, f))
        X.append(mel)

        # Encodage des labels
        Y_chiffre.append(chiffre)
        Y_speaker.append(Lspeaker.index(speaker))
        Y_occ.append(occ)


# Affichage des infos
print('X: ', np.shape(X), type(X))
print('Y_chiffre: ', np.shape(Y_chiffre), type(Y_chiffre))


X:  (3000, 128, 36, 1) <class 'list'>
Y_chiffre:  (3000,) <class 'list'>


## Séparation en données de Train et de Test et mise au format pour le réseau

Xtrain,Ytrain_chiffre,Ytrain_speaker

Xtest,Ytest_chiffre,Ytest_speaker  

° Pour chaque speaker et mot (chiffre) choisir N occurences pour le train et 50-N pour les tests. Il sera demandé de veiller à ce que les datasets Train et Test soient bien homogènes (à la fois dans les classes speaker et mot), et à vérifier l'homogénéité via la fonction OcurrencesClasses fournie.

In [ ]:

import numpy as np

N = 30

Xtrain = []
Ytrain_chiffre = []
Ytrain_speaker = []

Xtest = []
Ytest_chiffre = []
Ytest_speaker = []

X = np.array(X)
Y_chiffre = np.array(Y_chiffre)
Y_speaker = np.array(Y_speaker)
Y_occ = np.array(Y_occ)

#Pour chaque combinaison (speaker, chiffre)
for speaker in np.unique(Y_speaker):
    for chiffre in np.unique(Y_chiffre):
        # Indices correspondant à la bonne combi
        indices = np.where((Y_speaker == speaker) & (Y_chiffre == chiffre))[0]

        # Shuffle pour bien mélanger =avec homogénéité
        np.random.shuffle(indices)
        train = indices[:N]
        test = indices[N:]

        # dataset train
        for i in train:
            Xtrain.append(X[i])
            Ytrain_chiffre.append(Y_chiffre[i])
            Ytrain_speaker.append(Y_speaker[i])

        # dataset test
        for i in test:
            Xtest.append(X[i])
            Ytest_chiffre.append(Y_chiffre[i])
            Ytest_speaker.append(Y_speaker[i])

# i = 0
# for k in range(50,len(Y_chiffre),50):
#   p_chiffre = Y_chiffre[i:k]
#   p_speaker = Y_speaker[i:k]
#   p_melspec = X[i:k]
#   for j in range(len(p_chiffre)):
#     if j<N:
#       Xtrain.append(p_melspec[j])
#       Ytrain_chiffre.append(int(p_chiffre[j]))
#       Ytrain_speaker.append(p_speaker[j])
#     else:
#       Xtest.append(p_melspec[j])
#       Ytest_chiffre.append(int(p_chiffre[j]))
#       Ytest_speaker.append(p_speaker[j])
#   i = k



Xtrain1 = np.array(Xtrain)
Ytrain1_chiffre = np.array(Ytrain_chiffre)
Ytrain1_speaker = np.array(Ytrain_speaker)

Xtest1 = np.array(Xtest)
Ytest1_chiffre = np.array(Ytest_chiffre)
Ytest1_speaker = np.array(Ytest_speaker)


print('----------------------------')
print('Xtrain=', np.shape(Xtrain1))
print('occurences chiffre=', occurencesclasses(Ytrain1_chiffre, 10))   # 10 chiffres possibles
print('occurences speaker=', occurencesclasses(Ytrain1_speaker, len(Lspeaker)))
print('----------------------------')
print('Xtest=', np.shape(Xtest1))
print('occurences chiffre=', occurencesclasses(Ytest1_chiffre, 10))
print('occurences speaker=', occurencesclasses(Ytest1_speaker, len(Lspeaker)))


----------------------------
Xtrain= (1800, 128, 36, 1)
occurences chiffre= [180 180 180 180 180 180 180 180 180 180]
occurences speaker= [300 300 300 300 300 300]
----------------------------
Xtest= (1200, 128, 36, 1)
occurences chiffre= [120 120 120 120 120 120 120 120 120 120]
occurences speaker= [200 200 200 200 200 200]


Mise au format pour le réseau

In [ ]:
#############################################################################
# Ici les noms X et Y ont été utilisés par défaut, pour montrer la syntaxe
# Cette mise au format est à appliquer à chacun des "X" et "Y"
##############################################################################

# Transformation en array => La couche de convolution prend obligatoirement en entrée un tenseur de type array
X1=np.array(X);

# encode target values Y
#attention ! : vous devez bien comprendre ce que font ces lignes de code
#              ... et comprendre pourquoi il faudra utiliser "Y" et non "Y_nn" pour la matrice de confusion !
# Il est demandé d'expliquer cette mise en forme des données dans votre rapport
Y_nn = to_categorical(Y_chiffre)

print("X:"+ str(np.shape(X)) + " , Y_nn:" + str(np.shape(Y_nn)))


X:(3000, 128, 36, 1) , Y_nn:(3000, 10)


## Réseau mot : conception, entrainement, test, analyse des résultats (matrice de confusion)



Conception du réseau (architecture)

In [ ]:
Reseaudeneuronnes = Sequential()

# couche 1
Reseaudeneuronnes.add(Conv2D(32, (3,3), activation="relu", input_shape=(128, 36, 1)))
Reseaudeneuronnes.add(MaxPooling2D(2,2))
# couche 2
Reseaudeneuronnes.add(Conv2D(32, (3,3), activation="relu"))
Reseaudeneuronnes.add(MaxPooling2D(4,4))


# On passe en couche plate
#Couche 3
Reseaudeneuronnes.add(Flatten())
Reseaudeneuronnes.add(Dense(1000))
Reseaudeneuronnes.add(Dense(200))
Reseaudeneuronnes.add(Dense(40))

# Pour une sortie de 10 chiffres
#Couche 4
Reseaudeneuronnes.add(Dense(10, activation='softmax'))
# Reseaudeneuronnes.add(Dense(10))

#######################################################
# A Completer : Architecture de votre réseau
# vous pouvez changer le nom ReseauClassifMot si vous voulez
########################################################

# compile model
Reseaudeneuronnes.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
Reseaudeneuronnes.summary() # permet d'afficher l'architecture du réseaux, les noms et dimensions des couches, le nombre de paramètres du réseau.



/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Entraînement

In [ ]:
###################################################################
# ici il reste juste à changer le nom du fichier filesave,
# et les noms Xtrain, Xtest, Ytrain_nn, Ytest_nn si besoin

# Attention : lorsque vous copiez-collez cette cellule
# pour des futurs entrainements, pensez bien à changer
# le nom de filesave pour ne pas écraser un jeu de poids précédent
###################################################################

Y_trainchiffre1 = to_categorical(Ytrain1_chiffre)
Y_testchiffre1 = to_categorical(Ytest1_chiffre)

history = Reseaudeneuronnes.fit(Xtrain1, Y_trainchiffre1, epochs=20, batch_size=64,
                                validation_data=(Xtest1,Y_testchiffre1), verbose=1)
# evaluate model
_, acc = Reseaudeneuronnes.evaluate(Xtest1,to_categorical(Ytest1_chiffre))
print('> %.3f' % (acc * 100.0))
# save model parameters (sauvegarde du jeu de poids dans le Drive)
filesave="Enregistrement_entrainement"
Reseaudeneuronnes.save_weights("{}/{}.weights.h5".format(working_dir,filesave))
# Affichage des courbes d'entraînement (accuracy/loss de train et de test au cours des époques) et sauvegarde dans le working_dir
summarize_diagnostics(history,working_dir,filesave)


Une fois un jeu de poids sauvegardé dans le Drive, celui-ci pourra-être directement chargé (évitant ainsi de relancer un entraînement)

In [ ]:
# chargement d'un jeu de poids précédemment sauvegardé dans le dossier working_dir sous le nom filesave
fileload="Enregistrement_entrainement" # A renseigner

Reseaudeneuronnes.load_weights("{}/{}.weights.h5".format(working_dir,fileload))
# evaluate model
_, acc = Reseaudeneuronnes.evaluate(Xtest1,to_categorical(Ytest1_chiffre),verbose=0)
print('> %.3f' % (acc * 100.0))

> 80.083


In [ ]:
# Une idée : une fois l'entraînement fait et le jeu de poids validé (matrice de confusion)
# vous pourriez fusionner les 2 cellules ci-dessus (train et load) et ajouter une variable
# TrainReseau.  If TrainReseau==True => train & save weights Else load weights
# cela évitera un train inutile lors d'une prochaine compilation du notebook qui en plus écraserai le jeu de pois précédent.

Prédictions et matrice de confusion

In [ ]:
# Prédictions et  matrice de confusion
y_pred=Reseaudeneuronnes.predict(Xtest1)  # argument à compléter
Y_pred = np.argmax(y_pred, 1) # Decode Predicted labels

############################################
#Arguments de la fonction à compléter
# ainsi que titre de la matrice de confusion
# et nom des labels
############################################


cmMot=confusion_matrix(Ytest1_chiffre,Y_pred)
Mytitle='Matrice de confusion'
Names_Label= ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"] #["nom classe0","nom classe1", ...]

Mycmap='Blues'    #personnalisation de la couleur : https://matplotlib.org/2.0.2/examples/color/colormaps_reference.html  (par défaut = Blues)
plot_confusion_matrix(cmMot,title=Mytitle,target_names=Names_Label,cmap=Mycmap,normalize=False)
# normalize : 'False' = résultat en nombre d'occurences , 'True' = resultat en proportion pour chaque classe

38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step


## Réseau classification du speaker

Conception du réseau

In [ ]:
Reseaudeneuronnes_S = Sequential()

# couche 1
Reseaudeneuronnes_S.add(Conv2D(32, (3,3), activation="relu", input_shape=(128, 36, 1)))
Reseaudeneuronnes_S.add(MaxPooling2D(2,2))

Reseaudeneuronnes_S.add(Conv2D(32, (3,3), activation="relu"))
# Reseaudeneuronnes.add(MaxPooling2D(2,2))


# On passe en couche plate
Reseaudeneuronnes_S.add(Flatten())

Reseaudeneuronnes_S.add(Dense(1000))

Reseaudeneuronnes_S.add(Dense(200))

Reseaudeneuronnes_S.add(Dense(40))

# Pour une sortie de 10 chiffres
Reseaudeneuronnes_S.add(Dense(6, activation='softmax'))

#######################################################
# A Completer : Architecture de votre réseau
# vous pouvez changer le nom ReseauClassifMot si vous voulez
########################################################

# compile model
Reseaudeneuronnes_S.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
Reseaudeneuronnes_S.summary() # permet d'afficher l'architecture du réseaux, les noms et dimensions des couches, le nombre de paramètres du réseau.


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


On fait ensuite l'netrainement

On commence par faire que les noms correspondent à un chiffre

In [ ]:
print(Ytrain1_speaker)


[0 0 0 ... 5 5 5]


In [ ]:
###################################################################
# ici il reste juste à changer le nom du fichier filesave,
# et les noms Xtrain, Xtest, Ytrain_nn, Ytest_nn si besoin

# Attention : lorsque vous copiez-collez cette cellule
# pour des futurs entrainements, pensez bien à changer
# le nom de filesave pour ne pas écraser un jeu de poids précédent
###################################################################

Y_trainspeaker = to_categorical(Ytrain1_speaker)
Y_testspeaker = to_categorical(Ytest1_speaker)

history = Reseaudeneuronnes_S.fit(Xtrain1, Y_trainspeaker, epochs=20, batch_size=64,
                                validation_data=(Xtest1,Y_testspeaker), verbose=1)
# evaluate model
_, acc = Reseaudeneuronnes_S.evaluate(Xtest1,to_categorical(Ytest1_speaker))
print('> %.3f' % (acc * 100.0))
# save model parameters (sauvegarde du jeu de poids dans le Drive)
filesave="Enregistrement_entrainement_speaker"
Reseaudeneuronnes_S.save_weights("{}/{}.weights.h5".format(working_dir,filesave))
# Affichage des courbes d'entraînement (accuracy/loss de train et de test au cours des époques) et sauvegarde dans le working_dir
summarize_diagnostics(history,working_dir,filesave)

In [ ]:
# chargement d'un jeu de poids précédemment sauvegardé dans le dossier working_dir sous le nom filesave
fileload="Enregistrement_entrainement_speaker" # A renseigner

Reseaudeneuronnes_S.load_weights("{}/{}.weights.h5".format(working_dir,fileload))
# evaluate model
_, acc = Reseaudeneuronnes_S.evaluate(Xtest1,to_categorical(Ytest1_speaker),verbose=0)
print('> %.3f' % (acc * 100.0))

> 91.667


On fait ensuite la matrice de confusion

In [ ]:
# Prédictions et  matrice de confusion
y_pred_S=Reseaudeneuronnes_S.predict(Xtest1)  # argument à compléter
Y_pred_S = np.argmax(y_pred_S, 1) # Decode Predicted labels

############################################
#Arguments de la fonction à compléter
# ainsi que titre de la matrice de confusion
# et nom des labels
############################################


cmMot=confusion_matrix(Ytest1_speaker,Y_pred_S)
Mytitle='Matrice de confusion'
Names_Label= ["0", "1", "2", "3", "4", "5"] #["nom classe0","nom classe1", ...]

Mycmap='Blues'    #personnalisation de la couleur : https://matplotlib.org/2.0.2/examples/color/colormaps_reference.html  (par défaut = Blues)
plot_confusion_matrix(cmMot,title=Mytitle,target_names=Names_Label,cmap=Mycmap,normalize=False)
# normalize : 'False' = résultat en nombre d'occurences , 'True' = resultat en proportion pour chaque classe

38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step


---


# Votre étude : Création d'un deuxième réseau pour analyser différement le nombre, puis on concatène les deux avec un  troisième réseau pour être plus performant.



---

Voici l'exemple d'une FFT faites sur un seul signal audio. POur notre analyse on ne garde que les amplictude

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile

list1 = glob.glob(data_dir+"/recordings/*")

# Charger le fichier audio
sample_rate, data = wavfile.read(list1[1])

# # Si le fichier est stéréo, prendre un seul canal
# if len(data.shape) == 2:
#     data = data[:, 0]

# Taille de l'échantillon
N = len(data)

# Appliquer la FFT
fft_result = np.fft.fft(data)
frequencies = np.fft.fftfreq(N, 1 / sample_rate)

# Ne garder que la moitié positive du spectre
positive_freqs = frequencies[:N // 2]
magnitude = np.abs(fft_result)[:N // 2]

# Tracer le spectre de fréquences
plt.figure(figsize=(12, 6))
plt.plot(positive_freqs, magnitude)
plt.title('Spectre de fréquences de l\'audio')
plt.xlabel('Fréquence (Hz)')
plt.ylabel('Amplitude')
plt.grid()
plt.show()

print(list1)


# Création d'un fichier dans le drive où sont enregistré toutes les FFT des audio.

Fonction utile:
Voici les fonctions qui vont permettre de récupérer les fichiers audio, de les mettres de la même taille et de faire la FFT et conserver les spectres.

In [ ]:
import numpy as np
import librosa
import os
from tqdm import tqdm
from scipy.io import wavfile

# 1. Charger les fichiers audio

def recup_donnée(list_of_files):
  liste_audio = []
  for audio in list_of_files:
    sample_rate, data = wavfile.read(audio)
    liste_audio.append(data)

  return liste_audio


# 2. Uniformiser la durée des signaux (padding avec zéros)
def memetaille(audio_list):
    max1 = max(len(audio) for audio in audio_list)
    padded_list = [np.pad(audio, (0, max1 - len(audio)), mode='constant') for audio in audio_list]
    return padded_list

# 3. Appliquer la FFT et récupérer les amplitudes
def calculate_FFT(audio_list,ratio=10):
    fft_list = []
    for audio in audio_list:
      fft_result = np.fft.fft(audio)
      fft_result = np.abs(fft_result)[N // 2:]
      fft_list.append(fft_result)
    return fft_list

# 4. Uniformiser les longueurs de FFT (padding avec zéros à droite)
def spec_padding_fft(fft_list):
    max_len = max(len(fft) for fft in fft_list)
    padded_fft_list = [np.pad(fft, (0, max_len - len(fft)), mode='constant') for fft in fft_list]
    return padded_fft_list

# 5. Sauvegarder les vecteurs FFT

def save_fft_spec(path_spec_list, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    for fft, path in path_spec_list:
        filename = os.path.basename(path).replace('.wav', '.npy')
        np.save(os.path.join(save_dir, filename), fft)


compute_FFT = False

n = 3000  # Nombre de fichiers à traiter

if compute_FFT:
    print("Conversion des fichiers audio en spectrogrammes FFT")

    # Process the audio files: load, pad time, calculate FFT, pad FFT length
    audio_data = recup_donnée(list1)
    padded_audio_data = memetaille(audio_data)
    fft_results = calculate_FFT(padded_audio_data)
    # tqdm(padded_fft_results = spec_padding_fft(fft_results))

    # Calculate the magnitude of the FFT
    N = len(fft_results[1])
    #magnitude = np.abs(fft_results)[:N // 2]

    # Combine file paths with padded FFT results for saving
    # You need to pair the original file paths with the processed FFT data
    # Assuming the order is maintained through the processing steps
    path_spec_list = list(zip(fft_results,list1))

# enregistrement des spectogrammes

    # Save the padded FFT specs
    save_dir_fft = data_dir + "/spectogramme_FFT/"
    print(f"Saving FFT spectrograms to: {save_dir_fft}")
    # Using tqdm for progress bar during saving
    for _ in tqdm(map(lambda item: save_fft_spec([item], save_dir_fft), path_spec_list), total=n):
        pass

    print("FFT spectrograms saved.")


Séparation de données et récupération des informations sur le speaker et le nombre

In [ ]:

import os
import numpy as np


# Liste des speakers
Lspeaker = ['jackson', 'nicolas', 'theo', 'yweweler', 'george', 'lucas']


# Dossier contenant les fichiers FFT sauvegardés
dataFFT = os.path.join(data_dir, 'spectogramme_FFT')


# Initialisation des listes
X2 = []
Y_speaker = []
Y_chiffre = []
Y_occ = []


# Parcours des fichiers .npy dans le dossier
for f in tqdm(sorted(os.listdir(dataFFT))):  # tri pour reproductibilité
    if f.endswith('.npy'):
      # Exemple de nom : 6_theo_25.npy
      parts = f.split('.')[0].split('_')
      chiffre = int(parts[0])
      speaker = parts[1]
      occ = int(parts[2])


      # Chargement du spectre
      spectre = np.load(os.path.join(dataFFT, f))
      X2.append(spectre)


      # Encodage des labels
      Y_chiffre.append(chiffre)
      Y_speaker.append(Lspeaker.index(speaker))  # index du speaker
      Y_occ.append(occ)



# Conversion en tableaux numpy
X2 = np.array(X2)
Y_chiffre = np.array(Y_chiffre)
Y_speaker = np.array(Y_speaker)
Y_occ = np.array(Y_occ)


# Affichage des dimensions
print('X2 shape: ', X2.shape, type(X2))
print('Y_chiffre shape: ', Y_chiffre.shape, type(Y_chiffre))
print('Y_speaker shape: ', Y_speaker.shape, type(Y_speaker))
print('Y_occ shape: ', Y_occ.shape, type(Y_occ))


X2 shape:  (3000, 16381) <class 'numpy.ndarray'>
Y_chiffre shape:  (3000,) <class 'numpy.ndarray'>
Y_speaker shape:  (3000,) <class 'numpy.ndarray'>
Y_occ shape:  (3000,) <class 'numpy.ndarray'>


# **Séparation des données en train et en test**

Ici on redivise les FFT en deux parties, une pour entrainer le réseau et une pour le tester et voir sa précision. Avec le N = 30 on choisit de mettre 600 éléments en plus dans les valeurs d'entrainements.

In [ ]:
N = 30

Xtrain = []
Ytrain_chiffre = []
Ytrain_speaker = []

Xtest = []
Ytest_chiffre = []
Ytest_speaker = []


Y2_chiffre = np.array(Y_chiffre)
Y2_speaker = np.array(Y_speaker)
Y2_occ = np.array(Y_occ)

#Pour chaque combinaison (speaker, chiffre)
for speaker in np.unique(Y_speaker):
    for chiffre in np.unique(Y_chiffre):
        # Indices correspondant à la bonne combi
        indices = np.where((Y_speaker == speaker) & (Y_chiffre == chiffre))[0]

        # Shuffle pour bien mélanger
        np.random.shuffle(indices)
        train = indices[:N]
        test = indices[N:]

        # dataset train
        for i in train:
            Xtrain.append(X2[i])
            Ytrain_chiffre.append(Y_chiffre[i])
            Ytrain_speaker.append(Y_speaker[i])

        # dataset test
        for i in test:
            Xtest.append(X2[i])
            Ytest_chiffre.append(Y_chiffre[i])
            Ytest_speaker.append(Y_speaker[i])

Xtrain2 = np.array(Xtrain)
Ytrain2_chiffre = np.array(Ytrain_chiffre)
Ytrain2_speaker = np.array(Ytrain_speaker)

Xtest2 = np.array(Xtest)
Ytest2_chiffre = np.array(Ytest_chiffre)
Ytest2_speaker = np.array(Ytest_speaker)


print('----------------------------')
print('Xtrain=', np.shape(Xtrain2))
print('occurences chiffre=', occurencesclasses(Ytrain2_chiffre, 10))   # 10 chiffres possibles
print('occurences speaker=', occurencesclasses(Ytrain2_speaker, len(Lspeaker)))
print('----------------------------')
print('Xtest=', np.shape(Xtest))
print('occurences chiffre=', occurencesclasses(Ytest2_chiffre, 10))
print('occurences speaker=', occurencesclasses(Ytest2_speaker, len(Lspeaker)))



----------------------------
Xtrain= (1800, 16381)
occurences chiffre= [180 180 180 180 180 180 180 180 180 180]
occurences speaker= [300 300 300 300 300 300]
----------------------------
Xtest= (1200, 16381)
occurences chiffre= [120 120 120 120 120 120 120 120 120 120]
occurences speaker= [200 200 200 200 200 200]


On transforme ici les format pour pouvoir importer nos données dans le réseau de neurones. Ici X2 est converti en tableau numpy et Y_chiffre devient une liste de 0 et de 1. ( cette liste permet de créer les 10 noeuds de la fin du réseau attribuer à chaque nombre )

In [ ]:

X2=np.array(X2);

Y_nn2 = to_categorical(Y_chiffre)

print("X:"+ str(np.shape(X2)) + " , Y_nn:" + str(np.shape(Y_nn2)))

X:(3000, 16381) , Y_nn:(3000, 10)


# Création d'un nouveau réseau de neuronnes


On vient ici créer un deuxième réseau de neurones. Celui prend en entrée les amplitudes des transformée de fourier de chacun des audio et ressort une liste de à et 1 qui renvoie à un nombre.

In [ ]:
from tensorflow.keras.layers import AveragePooling1D, MaxPooling1D


Reseaudeneuronnes2 = Sequential()

# couche 1
Reseaudeneuronnes2.add(MaxPooling1D(pool_size=25, strides=None, padding='valid', input_shape=(16381,1)))
# couche 2
Reseaudeneuronnes2.add(Flatten())
#couche 3
Reseaudeneuronnes2.add(Dense(100))
#Couche 4
Reseaudeneuronnes2.add(Dense(40))
#Couche 5
# Pour une sortie de 10 chiffres
Reseaudeneuronnes2.add(Dense(10, activation='softmax'))
# Reseaudeneuronnes2.add(Dense(10))
# compile model
Reseaudeneuronnes2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
Reseaudeneuronnes2.summary() # permet d'afficher l'architecture du réseaux, les noms et dimensions des couches, le nombre de paramètres du réseau.

/usr/local/lib/python3.11/dist-packages/keras/src/layers/pooling/base_pooling.py:23: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(name=name, **kwargs)


**On entraine le réseau et on le test**

In [ ]:


Y2_trainchiffre = to_categorical(Ytrain2_chiffre)
Y2_testchiffre = to_categorical(Ytest2_chiffre)

history = Reseaudeneuronnes2.fit(Xtrain2, Y2_trainchiffre, epochs=20, batch_size=64,
                                validation_data=(Xtest2,Y2_testchiffre), verbose=1)

_, acc = Reseaudeneuronnes2.evaluate(Xtest2,to_categorical(Ytest2_chiffre))
print('> %.3f' % (acc * 100.0))

# save model parameters (sauvegarde du jeu de poids dans le Drive)
filesave="Enregistrement_entrainement_Réseau2"

Reseaudeneuronnes2.save_weights("{}/{}.weights.h5".format(working_dir,filesave))
# Affichage des courbes d'entraînement (accuracy/loss de train et de test au cours des époques) et sauvegarde dans le working_dir
summarize_diagnostics(history,working_dir,filesave)

In [ ]:
# chargement d'un jeu de poids précédemment sauvegardé dans le dossier working_dir sous le nom filesave
fileload="Enregistrement_entrainement_Réseau2"

Reseaudeneuronnes2.load_weights("{}/{}.weights.h5".format(working_dir,fileload))
# evaluate model
_, acc = Reseaudeneuronnes2.evaluate(Xtest2,to_categorical(Ytest2_chiffre),verbose=0)
print('> %.3f' % (acc * 100.0))

> 78.333


Pour voir la précision de notre réseau on trace la matrice de confusion sur le résultat donné par le réseau sur les valeurs de test.

In [ ]:

# Prédictions et  matrice de confusion
y2_pred=Reseaudeneuronnes2.predict(Xtest2)  # argument à compléter
Y2_pred = np.argmax(y2_pred, 1) # Decode Predicted labels

cmMot=confusion_matrix(Ytest2_chiffre,Y2_pred)
Mytitle='Matrice de confusion'
Names_Label= ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"] #["nom classe0","nom classe1", ...]

Mycmap='Blues'
plot_confusion_matrix(cmMot,title=Mytitle,target_names=Names_Label,cmap=Mycmap,normalize=False)
# normalize : 'False' = résultat en nombre d'occurences , 'True' = resultat en proportion pour chaque classe

38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step




---

 # On va ensuite créer un troisième réseau qui nous permet de concaténer en entrée la sortie de nos deux précédent réseau. L'objectif de ce réseau est d'obtenir une polus grande précision que avec un réseau unique.

On commenxe par diviser les données et à enregistrer deux listes : celle avec les spectogrammes et celle avec les melspectogrammes.

In [ ]:

import numpy as np
import librosa
import os
from tqdm import tqdm
from scipy.io import wavfile

# Liste des speakers
Lspeaker = ['jackson', 'nicolas', 'theo', 'yweweler', 'george', 'lucas']

working_dir = 'drive/MyDrive/<your_folder>'
data_dir=working_dir+"/Datasets"

# Dossier contenant les fichiers FFT sauvegardés
dataFFT = os.path.join(data_dir, 'spectogramme_FFT')
datamelspec = data_dir + '/melspectrograms'


# Initialisation des listes
X1 = []
X2 = []
Y_speaker = []
Y_chiffre = []
Y_occ = []

# Parcours des fichiers .npy dans le dossier FFT
for f in tqdm(os.listdir(dataFFT)):  # tri pour reproductibilité
    if f.endswith('.npy'):
      # Exemple de nom : 6_theo_25.npy
      parts = f.split('.')[0].split('_')
      chiffre = int(parts[0])
      speaker = parts[1]
      occ = int(parts[2])

      # Chargement du spectre
      spectre = np.load(os.path.join(dataFFT, f))
      X2.append(spectre)
      # Encodage des labels
      Y_chiffre.append(chiffre)
      Y_speaker.append(Lspeaker.index(speaker))  # index du speaker
      Y_occ.append(occ)

# Parcours des fichiers .npy dans le dossier melspec
for f in os.listdir(datamelspec):
    if f.endswith('.npy'):
        mel = np.load(os.path.join(datamelspec, f))
        X1.append(mel)

# Conversion en tableaux numpy
X2 = np.array(X2)
X1 = np.array(X1)
Y_chiffre = np.array(Y_chiffre)
Y_speaker = np.array(Y_speaker)
Y_occ = np.array(Y_occ)

# Affichage des dimensions
print('X2 shape: ', X2.shape, type(X2))
print('X1 shape: ', X1.shape, type(X1))
print('Y_chiffre shape: ', Y_chiffre.shape, type(Y_chiffre))
print('Y_speaker shape: ', Y_speaker.shape, type(Y_speaker))
print('Y_occ shape: ', Y_occ.shape, type(Y_occ))

X2 shape:  (3000, 16381) <class 'numpy.ndarray'>
X1 shape:  (3000, 128, 36, 1) <class 'numpy.ndarray'>
Y_chiffre shape:  (3000,) <class 'numpy.ndarray'>
Y_speaker shape:  (3000,) <class 'numpy.ndarray'>
Y_occ shape:  (3000,) <class 'numpy.ndarray'>


Ici on réparti notre base de donnée entre les valeurs d'entrainement et les valeurs de test.

In [ ]:
N = 30
Xtrain1 = []
Xtest1 = []

Xtrain2 = []
Ytrain_chiffre = []
Ytrain_speaker = []

Xtest2 = []
Ytest_chiffre = []
Ytest_speaker = []


Y_chiffre = np.array(Y_chiffre)
Y_speaker = np.array(Y_speaker)
Y_occ = np.array(Y_occ)

#Pour chaque combinaison (speaker, chiffre)
for speaker in np.unique(Y_speaker):
    for chiffre in np.unique(Y_chiffre):
        # Indices correspondant à la bonne combi
        indices = np.where((Y_speaker == speaker) & (Y_chiffre == chiffre))[0]

        # Shuffle pour bien mélanger
        np.random.shuffle(indices)
        train = indices[:N]
        test = indices[N:]

        # dataset train
        for i in train:
            Xtrain2.append(X2[i])
            Xtrain1.append(X1[i])
            Ytrain_chiffre.append(Y_chiffre[i])
            Ytrain_speaker.append(Y_speaker[i])

        # dataset test
        for i in test:
            Xtest2.append(X2[i])
            Xtest1.append(X1[i])
            Ytest_chiffre.append(Y_chiffre[i])
            Ytest_speaker.append(Y_speaker[i])

Xtrain2 = np.array(Xtrain2)
Xtrain1 = np.array(Xtrain1)
Ytrain_chiffre = np.array(Ytrain_chiffre)
Ytrain_speaker = np.array(Ytrain_speaker)

Xtest1 = np.array(Xtest1)
Xtest2 = np.array(Xtest2)
Ytest_chiffre = np.array(Ytest_chiffre)
Ytest_speaker = np.array(Ytest_speaker)


print('----------------------------')
print('Xtrain2=', np.shape(Xtrain2))
print('Xtrain1=', np.shape(Xtrain1))
print('occurences chiffre=', occurencesclasses(Ytrain_chiffre, 10))   # 10 chiffres possibles
print('occurences speaker=', occurencesclasses(Ytrain_speaker, len(Lspeaker)))
print('----------------------------')
print('Xtest1=', np.shape(Xtest1))
print('Xtest2=', np.shape(Xtest2))
print('occurences chiffre=', occurencesclasses(Ytest_chiffre, 10))
print('occurences speaker=', occurencesclasses(Ytest_speaker, len(Lspeaker)))



----------------------------
Xtrain2= (1800, 16381)
Xtrain1= (1800, 128, 36, 1)
occurences chiffre= [180 180 180 180 180 180 180 180 180 180]
occurences speaker= [300 300 300 300 300 300]
----------------------------
Xtest1= (1200, 128, 36, 1)
Xtest2= (1200, 16381)
occurences chiffre= [120 120 120 120 120 120 120 120 120 120]
occurences speaker= [200 200 200 200 200 200]


On créer ensuite notre troisième réseau de neurones

In [ ]:

model = Sequential()
# model.add(MaxPooling1D(pool_size=1, strides=None, padding='valid', input_shape=(20,1)))
# couche 2
model.add(Flatten(input_shape=(20,1)))



model.add(Dense(10,activation="softmax"))


model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary() # permet d'afficher l'architecture du réseaux, les noms et dimensions des couches, le n



On retravaille nos données d'entrée pour qu'elles correspondent à l'entrée de notre troisième réseau.

In [ ]:
Y_trainchiffre = to_categorical(Ytrain_chiffre)
Y_testchiffre = to_categorical(Ytest_chiffre)

def ajout(X1,X2):
  XR1=Reseaudeneuronnes.predict(X1)
  print(XR1.shape)
  XR2=Reseaudeneuronnes2.predict(X2)
  print(XR2.shape)
  X1=np.array(XR1)
  X2=np.array(XR2)
  return np.concatenate([X1,X2],axis=1)


Xfused = ajout(Xtrain1, Xtrain2)
print("Shape ajout Xtrain:", Xfused.shape)

# Ici Xtrain1 représente le Xtrain1 de la verison melspec et Xtrain2 la version du spectre
history = model.fit(ajout(Xtrain1,Xtrain2), Y_trainchiffre, epochs=40, batch_size=64,
                                validation_data=(ajout(Xtest1, Xtest2), Y_testchiffre)
, verbose=1)

# evaluate model
_, acc = model.evaluate(ajout(Xtest1,Xtest2),to_categorical(Ytest_chiffre))
print('> %.3f' % (acc * 100.0))


# save model parameters (sauvegarde du jeu de poids dans le Drive)
filesave="Enregistrement_entrainement_final_troisième_réseau"

model.save_weights("{}/{}.weights.h5".format(working_dir,filesave))
# Affichage des courbes d'entraînement (accuracy/loss de train et de test au cours des époques) et sauvegarde dans le working_dir
summarize_diagnostics(history,working_dir,filesave)


On enregistre ensuite les poids de ce réseau

In [ ]:
# chargement d'un jeu de poids précédemment sauvegardé dans le dossier working_dir sous le nom filesave
fileload="Enregistrement_entrainement_final_troisième_réseau"


# model.load_weights("{}/{}.weights.h5".format(working_dir,fileload))

# evaluate model
_, acc = model.evaluate(ajout(Xtest1,Xtest2),to_categorical(Ytest_chiffre),verbose=0)
print('> %.3f' % (acc * 100.0))

38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
(1200, 10)
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
(1200, 10)
> 83.750


ON trace ensuite la matrice de confusion pour visualiser la prcision de notre réseau

In [ ]:
# Prédictions et  matrice de confusion
y_pred=model.predict(ajout(Xtest1,Xtest2))  # argument à compléter
Y_pred = np.argmax(y_pred, 1) # Decode Predicted labels


cmMot=confusion_matrix(Ytest_chiffre,Y_pred)
Mytitle='Matrice de confusion'
Names_Label= ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"] #["nom classe0","nom classe1", ...]

acc1 = np.mean(np.argmax(Reseaudeneuronnes.predict(Xtest1), axis=1) == Ytest_chiffre)
acc2 = np.mean(np.argmax(Reseaudeneuronnes2.predict(Xtest2), axis=1) == Ytest_chiffre)


print("Xtrain1 shape:", Xtrain1.shape)
print("Xtrain2 shape:", Xtrain2.shape)
print("Ytrain_chiffre shape:", Ytrain_chiffre)


Mycmap='Blues'    #personnalisation de la couleur : https://matplotlib.org/2.0.2/examples/color/colormaps_reference.html  (par défaut = Blues)
plot_confusion_matrix(cmMot,title=Mytitle,target_names=Names_Label,cmap=Mycmap,normalize=False)
# normalize : 'False' = résultat en nombre d'occurences , 'True' = resultat en proportion pour chaque classe

38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
(1200, 10)
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
(1200, 10)
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Xtrain1 shape: (1800, 128, 36, 1)
Xtrain2 shape: (1800, 16381)
Ytrain_chiffre shape: [0 0 0 ... 9 9 9]
